In [145]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score, accuracy_score

In [146]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
!wget $data -O data-hw-3.csv

--2026-10-12 05:55:25--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘data-hw-3.csv’

data-hw-3.csv       100%[===================>] 272.21K  --.-KB/s    in 0.01s   

2026-10-12 05:55:25 (23.9 MB/s) - ‘data-hw-3.csv’ saved [278746/278746]



In [147]:
df = pd.read_csv('data-hw-3.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [148]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [149]:
categorical = list(df.select_dtypes(include=['object', 'string']).columns)
numerical = list(df.select_dtypes(include=['int64', 'float64']).columns)

for c in categorical:
    df[c] = df[c].fillna('NA')
for n in numerical:
    df[n] = df[n].fillna(0.0)


df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [150]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [151]:
df[numerical].corrwith(df.converted)

annual_income               0.189652
number_of_courses_viewed    0.395038
interaction_count           0.448624
lead_score                  0.483246
converted                   1.000000
dtype: float64

In [152]:
df.interaction_count.corr(df.lead_score)

np.float64(0.9157457980418677)

In [153]:
df.number_of_courses_viewed.corr(df.lead_score)

np.float64(0.7572042392104887)

In [154]:
df.number_of_courses_viewed.corr(df.interaction_count)

np.float64(0.7216090038937009)

In [155]:
df.annual_income.corr(df.interaction_count)

np.float64(0.12284235135955954)

In [156]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [157]:
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [158]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [159]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [160]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [161]:
numerical = [n for n in numerical if n != 'converted']

In [166]:
cat_cols = ['industry', 'location', 'lead_source', 'employment_status']

def mi(col):
    return mutual_info_score(df_train[col], y_train)

mi_scores = {col: round(mi(col), 2) for col in cat_cols}
mi_scores


{'industry': 0.0,
 'location': 0.0,
 'lead_source': 0.03,
 'employment_status': 0.02}

In [162]:
for c in categorical:
    df_train[c] = df_train[c].fillna('NA')
    df_val[c] = df_val[c].fillna('NA')

for n in numerical:
    df_train[n] = df_train[n].fillna(0.0)
    df_val[n] = df_val[n].fillna(0.0)

In [163]:
features = cat_cols + num_cols
train_dicts = df_train[features].to_dict(orient='records')
val_dicts = df_val[features].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
original_accuracy = accuracy_score(y_val, y_pred)
round(original_accuracy, 2)


0.65

In [164]:
for f in features:
    subset = [c for c in features if c != f]

    dv_f = DictVectorizer(sparse=False)
    X_tr = dv_f.fit_transform(df_train[subset].to_dict(orient='records'))
    X_vl = dv_f.transform(df_val[subset].to_dict(orient='records'))

    m = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    m.fit(X_tr, y_train)

    acc = accuracy_score(y_val, m.predict(X_vl))
    print(f, acc, original_accuracy - acc)

lead_source 0.642 0.0030000000000000027
industry 0.645 0.0
employment_status 0.645 0.0
location 0.645 0.0
annual_income 0.724 -0.07899999999999996
number_of_courses_viewed 0.643 0.0020000000000000018
interaction_count 0.601 0.04400000000000004
lead_score 0.644 0.0010000000000000009


In [165]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    m = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    m.fit(X_train, y_train)

    acc = accuracy_score(y_val, m.predict(X_val))
    print(C, round(acc, 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
